In [ ]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os
from requests.auth import HTTPBasicAuth

load_dotenv()

# Replace with your actual values
retainer_url = os.getenv("retainer_url")
username = os.getenv("username")
password = os.getenv("password")
channel_id = os.getenv("channel_id")

def get_odata_dataframe(url, username, password):
    try:
        response = requests.get(url, auth=HTTPBasicAuth(username, password))
        response.raise_for_status()
        data = response.json()

        if 'value' in data:
            df = pd.DataFrame(data['value'])
            print("✅ Data successfully loaded into DataFrame")
            return df
        else:
            raise ValueError("No 'value' key found in JSON response")
    except Exception as e:
        print(f"❌ Error: {e}")
        return pd.DataFrame()
    
def get_odata_dataframe(url, username, password):
    try:
        response = requests.get(url, auth=HTTPBasicAuth(username, password))
        response.raise_for_status()
        data = response.json()

        if 'value' in data:
            df = pd.DataFrame(data['value'])
            print("✅ Data successfully loaded into DataFrame")
            return df
        else:
            raise ValueError("No 'value' key found in JSON response")
    except Exception as e:
        print(f"❌ Error: {e}")
        return pd.DataFrame()

# Load and preview data
df_retainer = get_odata_dataframe(retainer_url, username, password)


In [ ]:
df_retainer.head()

In [ ]:
filtered_retainer_df = df_retainer[
    (df_retainer["RETAINER_STATE"] == "Open") &
    (~df_retainer["NAME"].str.contains("Flume", case=False, na=False)) &
    (~df_retainer["NAME"].str.contains("Elaine'", case=False, na=False)) &
    (~df_retainer["NAME"].str.contains("Lee", case=False, na=False))
]

In [ ]:
filtered_retainer_df.head()

In [ ]:
import numpy as np

filtered_retainer_df["USEDVALUE"] = filtered_retainer_df["USEDVALUE"].astype(float)
filtered_retainer_df["INVOICEDVALUE"] = filtered_retainer_df["INVOICEDVALUE"].astype(float)
filtered_retainer_df["UTILIZATION"] = (
    filtered_retainer_df["USEDVALUE"] / filtered_retainer_df["INVOICEDVALUE"]
).replace([np.inf, -np.inf], np.nan).fillna(0) * 100


In [ ]:
filtered_retainer_df["UTILIZATION"]

In [ ]:
df_retainer_50_check = filtered_retainer_df[(filtered_retainer_df['UTILIZATION'] > 50) & (filtered_retainer_df['UTILIZATION'] < 80)]
df_retainer_80_check = filtered_retainer_df[filtered_retainer_df['UTILIZATION'] > 80]

In [ ]:
df_retainer_50_check.head()

In [ ]:
import requests
import pandas as pd

# 🔑 Replace with your actual bot token
slack_token = os.getenv("slack_token")
headers = {
    "Authorization": f"Bearer {slack_token}"
}

def fetch_slack_users():
    url = "https://slack.com/api/users.list"
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        data = response.json()

        if not data.get("ok"):
            raise Exception(f"Slack API Error: {data.get('error')}")

        # Extract user ID and real name
        users = [
            {
                "user_id": member["id"],
                "real_name": member.get("real_name", ""),
                "email": member.get("profile", {}).get("email", ""),
                "job_title": member.get("profile", {}).get("title", ""),
                "is_bot": member.get("is_bot", False),
                "deleted": member.get("deleted", False)
            }
            for member in data["members"]
        ]

        df = pd.DataFrame(users)
        print("✅ Slack users pulled into DataFrame")
        return df

    except Exception as e:
        print(f"❌ Error: {e}")
        return pd.DataFrame()

# Run the function
df_slack_users = fetch_slack_users()

# Preview the result
df_slack = df_slack_users[
        (~df_slack_users['is_bot']) & (~df_slack_users['deleted'])
    ]
df_slack['email'] = df_slack['email'].str.lower()
df_slack.head()


In [ ]:
def fetch_slack_channels():
    url = "https://slack.com/api/conversations.list"
    params = {
        "exclude_archived": True,
        "limit": 1000,  # adjust if needed
        "types": "public_channel,private_channel"
    }

    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        data = response.json()

        if not data.get("ok"):
            raise Exception(f"Slack API Error: {data.get('error')}")

        # Extract relevant info
        channels = [
            {
                "channel_id": ch["id"],
                "name": ch["name"],
                "is_private": ch["is_private"],
                "member_count": ch.get("num_members")
            }
            for ch in data["channels"]
        ]

        df_channels = pd.DataFrame(channels)
        print("✅ Slack channels pulled into DataFrame")
        return df_channels

    except Exception as e:
        print(f"❌ Error fetching Slack channels: {e}")
        return pd.DataFrame()


In [ ]:
import pandas as pd
import numpy as np
import datetime

# === STEP 1: Calculate utilisation safely ====================================
# Assumes df_invoice has at least: USEDVALUE, INVOICEDVALUE and a name field
# e.g. 'DEPOSIT_NAME' or 'CLIENT'. Adjust the column name below if needed.

# Convert to numeric just in case
filtered_retainer_df['USEDVALUE'] = pd.to_numeric(filtered_retainer_df['USEDVALUE'], errors='coerce')
filtered_retainer_df['INVOICEDVALUE'] = pd.to_numeric(filtered_retainer_df['INVOICEDVALUE'], errors='coerce')

# Utilisation as a percentage, handling div-by-zero, NaN and inf
filtered_retainer_df['UTILIZATION'] = (
    filtered_retainer_df['USEDVALUE'] / filtered_retainer_df['INVOICEDVALUE']
).replace([np.inf, -np.inf], np.nan).fillna(0) * 100

# === STEP 2: Filter retainers that have reached 50% of budget ================
# Here I’m using 50–80% as the “keep an eye on these” band.
# Change 80 to 100 if you want ">= 50 and < 100" instead.
df_retainer_50_check = filtered_retainer_df[
    (filtered_retainer_df['UTILIZATION'] >= 50) &
    (filtered_retainer_df['UTILIZATION'] < 80)
]

# === STEP 3: Build the Slack message =========================================
today = datetime.date.today()

retainer_message = (
    "The following retainer deposits have now reached 50% of their budget."
    "\nPlease keep an eye on them to help prevent any overspending."
    "\nStaying on top of these ensures everything runs smoothly, so thank you in advance.\n\n"
    f"As at {today.strftime('%A, %d %B %Y')}:\n"
)

if df_retainer_50_check.empty:
    retainer_message += "\n_No retainers have reached 50% of their budget yet._"
else:
    lines = []
    for _, row in df_retainer_50_check.iterrows():
        # Change 'DEPOSIT_NAME' to whatever identifies the retainer (e.g. 'CLIENT') ENTITY_CLIENT_NAME
        name = row.get('ENTITY_CLIENT_NAME', row.get('CLIENT', 'Unknown retainer'))
        retainer_name = row.get('NAME', row.get('CLIENT', 'Unknown retainer'))
        util = row['UTILIZATION']
        used = row['USEDVALUE']
        budget = row['INVOICEDVALUE']

        lines.append(
            f"• {name}, {retainer_name}: *{util:.1f}%* used "
            f"(R{used:,.0f} of R{budget:,.0f})"
        )

    retainer_message += "\n".join(lines)

# === STEP 4: Print / send =====================================================
print(retainer_message)
# send to Slack here using your existing Slack client


In [ ]:
def send_message_as_user(channel_id, message, user_token):
    url = "https://slack.com/api/chat.postMessage"
    headers = {
        "Authorization": f"Bearer {user_token}",
        "Content-type": "application/json"
    }
    payload = {
        "channel": channel_id,
        "text": message
    }
    response = requests.post(url, headers=headers, json=payload)
    if response.status_code == 200 and response.json().get("ok"):
        print("✅ Message sent successfully")
    else:
        print("❌ Failed to send message:", response.text)

In [ ]:
send_message_as_user(
    channel_id=channel_id,
    message=retainer_message,
    user_token=slack_token
)

In [ ]:
import pandas as pd
import numpy as np
import datetime

# === STEP 1: Calculate utilisation safely ====================================
# Assumes df_invoice has at least: USEDVALUE, INVOICEDVALUE and a name field
# e.g. 'DEPOSIT_NAME' or 'CLIENT'. Adjust the column name below if needed.

# Convert to numeric just in case
filtered_retainer_df['USEDVALUE'] = pd.to_numeric(filtered_retainer_df['USEDVALUE'], errors='coerce')
filtered_retainer_df['INVOICEDVALUE'] = pd.to_numeric(filtered_retainer_df['INVOICEDVALUE'], errors='coerce')

# Utilisation as a percentage, handling div-by-zero, NaN and inf
filtered_retainer_df['UTILIZATION'] = (
    filtered_retainer_df['USEDVALUE'] / filtered_retainer_df['INVOICEDVALUE']
).replace([np.inf, -np.inf], np.nan).fillna(0) * 100

# === STEP 2: Filter retainers that have reached 50% of budget ================
# Here I’m using 50–80% as the “keep an eye on these” band.
# Change 80 to 100 if you want ">= 50 and < 100" instead.
df_retainer_80_check = filtered_retainer_df[
    (filtered_retainer_df['UTILIZATION'] >= 80) &
    (filtered_retainer_df['UTILIZATION'] < 100)
]

# === STEP 3: Build the Slack message =========================================
today = datetime.date.today()

retainer_message = (
    "The following retainer deposits have now *surpassed 80%* of their budget."
    "\nPlease keep an eye on them to help prevent any overspending."
    "\nStaying on top of these ensures everything runs smoothly, so thank you in advance.\n\n"
    f"As at {today.strftime('%A, %d %B %Y')}:\n"
)

if df_retainer_50_check.empty:
    retainer_message += "\n_No retainers have reached 50% of their budget yet._"
else:
    lines = []
    for _, row in df_retainer_80_check.iterrows():
        # Change 'DEPOSIT_NAME' to whatever identifies the retainer (e.g. 'CLIENT')
        name = row.get('NAME', row.get('CLIENT', 'Unknown retainer'))
        util = row['UTILIZATION']
        used = row['USEDVALUE']
        budget = row['INVOICEDVALUE']

        lines.append(
            f"• {name}: *{util:.1f}%* used "
            f"(R{used:,.0f} of R{budget:,.0f})"
        )

    retainer_message += "\n".join(lines)

# === STEP 4: Print / send =====================================================
print(retainer_message)
# send to Slack here using your existing Slack client


In [ ]:
def send_message_as_user(channel_id, message, user_token):
    url = "https://slack.com/api/chat.postMessage"
    headers = {
        "Authorization": f"Bearer {user_token}",
        "Content-type": "application/json"
    }
    payload = {
        "channel": channel_id,
        "text": message
    }
    response = requests.post(url, headers=headers, json=payload)
    if response.status_code == 200 and response.json().get("ok"):
        print("✅ Message sent successfully")
    else:
        print("❌ Failed to send message:", response.text)

In [ ]:
send_message_as_user(
    channel_id=channel_id,
    message=retainer_message,
    user_token=slack_token
)

Markdown

In [ ]:
#code